In [10]:
import shapely.geometry as sg
import glob
import re
from image_analysis_functions import extract_unique

import ee 
import geemap
import geopandas as gpd
import pandas as pd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

In [11]:
overlap_directory = './data/overlap_dates_for_roi/'
overlap_file_list = glob.glob(f'{overlap_directory}/YKD_*.shp')

def extract_roi_name(file_path: str) -> str:
    # Regex pattern: capture text that includes ROI letters, "_sub", and one or more digits
    pattern = re.compile(r".*/([A-Z]+_sub\d+)_overlap_dates\.shp$")
    match = pattern.search(file_path)
    if match:
        return match.group(1)
    else:
        return None
    
footprints = []

for f in overlap_file_list:
    gdf = gpd.read_file(f)
    roi_name = extract_roi_name(f)
    gdf['roi_name'] = roi_name
    footprints.append(gdf)

footprints = pd.concat(footprints, ignore_index=True)
footprints.to_crs(crs='EPSG:4326', inplace=True)


In [12]:
def geom_to_ee_polygon(geom):
    """
    Converts a geopandas geometry into an EE Polygon
    """
    coords = geom.exterior.coords
    coords_list = [[x, y] for x, y in coords]
    polygon = ee.Geometry.Polygon(coords_list)

    return polygon

In [13]:
def fetch_ls_angles(row: gpd.GeoSeries):
    
    asset_string = 'LANDSAT/LC08/C02/T1_TOA'
    date = row['date']
    start_date = ee.Date(date)
    end_date = start_date.advance(1, 'day')
    
    polygon = geom_to_ee_polygon(row.geometry)

    # Filter the collection by date and location.
    collection = ee.ImageCollection(asset_string) \
        .filterDate(start_date, end_date) \
        .filterBounds(polygon) \
        .select(['SAA', 'SZA'])

    # For each image in the collection get the mean, min and max of 'SAA' and 'SZA' bands
    reducer = ee.Reducer.mean() \
        .combine(ee.Reducer.minMax(), sharedInputs=True)
    
    def compute_stats(img):
        stats = img.reduceRegion(
            reducer=reducer,
            geometry=polygon,
            scale=30,
            maxPixels=1e13
        )
        return img.set(stats)
    
    collection = collection.map(compute_stats)

    collection_size = collection.size().getInfo()
    print(f'{collection_size} LS8 images in footprint')

    if collection_size > 0:
        image_list = collection.toList(collection_size)
        all_stats = []
        all_attrs = []

        for i in range(collection_size):
            img = ee.Image(image_list.get(i))
            stats = img.reduceRegion(
                reducer=reducer, 
                geometry=polygon,
                scale=30,
                maxPixels=1e13
            ).getInfo()
            all_stats.append(stats)

            attrs_sun_azimuth = img.get('SUN_AZIMUTH').getInfo()
            attrs_sun_zenith = img.get('SUN_ELEVATION').getInfo()

            attrs = {'attrs_sun_azimuth': attrs_sun_azimuth,
                     'attrs_sun_zenith': attrs_sun_zenith}
            
            all_attrs.append(attrs)

        return collection_size, all_stats, all_attrs
    
    else: 
        return 0, None, None


In [14]:
def fetch_s2_angles(row: gpd.GeoSeries):
    
    asset_string = "COPERNICUS/S2_HARMONIZED"
    date = row['date']
    start_date = ee.Date(date)
    end_date = start_date.advance(1, 'day')
    
    polygon = geom_to_ee_polygon(row.geometry)

    # # Filter the collection by date and location.
    collection = ee.ImageCollection(asset_string) \
        .filterDate(start_date, end_date) \
        .filterBounds(polygon)
    
    collection_size = collection.size().getInfo()
    print(f'{collection_size} S2 images in footprint')
    if collection_size > 0:
        image_list = collection.toList(collection_size)
        all_attrs = []

        for i in range(collection_size):
            img = ee.Image(image_list.get(i))
            sun_azimuth = img.get('MEAN_SOLAR_AZIMUTH_ANGLE').getInfo()
            sun_zenith = img.get('MEAN_SOLAR_ZENITH_ANGLE').getInfo()

            attrs = {'attrs_sun_azimuth': sun_azimuth,
                     'attrs_sun_zenith': sun_zenith}
            all_attrs.append(attrs)
            
        return collection_size, all_attrs
    
    else: 
        return 0, None


In [15]:
def rescale_ls_stats(stats_list: list):
    """Rescales all landsat stats to correspond to angle values""" 
    scaled_list = []
    for i in stats_list:
        stats_dict = i  # Get first dictionary from list
        scaled_dict = {}
        for key, value in stats_dict.items():
            scaled_dict[key] = value/100
        scaled_list.append(scaled_dict)
    return scaled_list

In [16]:
angle_info = []
for idx, row in footprints.iterrows():
    new = row.copy()
    img_cnt, angle_stats, angle_attrs = fetch_ls_angles(row)
    angle_stats_rescaled = rescale_ls_stats(angle_stats)
    new['ls_img_cnt'] = img_cnt
    new['ls_angle_stats'] = angle_stats_rescaled
    # Rescale the ls angle stats
    new['ls_angle_attrs'] = angle_attrs

    img_cnt, angle_attrs = fetch_s2_angles(row)
    new['s2_img_cnt'] = img_cnt
    new['s2_angle_attrs'] = angle_attrs

    angle_info.append(new)

footprints_with_angles = pd.DataFrame(angle_info)


1 LS8 images in footprint
4 S2 images in footprint
2 LS8 images in footprint
4 S2 images in footprint
1 LS8 images in footprint
4 S2 images in footprint
2 LS8 images in footprint
4 S2 images in footprint
1 LS8 images in footprint
4 S2 images in footprint
1 LS8 images in footprint
4 S2 images in footprint
2 LS8 images in footprint
4 S2 images in footprint


KeyboardInterrupt: 

In [17]:
footprints_with_angles = pd.DataFrame(angle_info)

In [18]:
#print(footprints_with_angles['s2_angle_attrs'].iloc[0])

[{'attrs_sun_azimuth': 173.97611756, 'attrs_sun_zenith': 40.2152483618}, {'attrs_sun_azimuth': 174.116990968, 'attrs_sun_zenith': 41.1055674335}, {'attrs_sun_azimuth': 174.669888574, 'attrs_sun_zenith': 40.1958929132}, {'attrs_sun_azimuth': 174.574284935, 'attrs_sun_zenith': 41.0944528582}]
